# Tessera | Portable Test Center

**Kullanım:** Bu notebook'u Google Colab'da aç → `Runtime / Çalışma zamanı → Run all / Tümünü çalıştır`.

Kod hücreleri gizli tutulur. Testler otomatik olarak hazırlanır, çalıştırılır ve sonuçlar tek bir raporda toplanır. Başarısızlık halinde aşağıdaki **Hata raporu** bölümünü kopyalayıp ChatGPT'ye gönder.

---

### Bu test neyi doğrular?
- Gerçek Tessera kaynak dosyalarının Linux üzerinde Win32 mock ortamında derlenmesini
- GCC ve Clang ile katı uyarı kontrollerini
- ASan / UBSan ile taşma, bellek ve tanımsız davranış kontrollerini

> **Sınır:** Bunlar Windows üzerinde gerçek hook, unwind, DLL lifecycle, MSVC veya Detours doğrulaması değildir. Windows testleri ayrıca gereklidir.

### İlk kullanım — erişim
Colab sol panelindeki **Secrets (anahtar)** bölümüne `GH_TOKEN` ekle. Fine-grained token yalnızca `Jue-dev/Tessera` için **Contents: Read-only** olmalı. Notebook erişimini etkinleştir. Token'ı hücrelere yazma.

### Test edilen sürüm
Notebook test başlamadan önce branch'in güncel commit SHA'sını kaydeder; rapor bu sürüme bağlanır.

In [ ]:
#@title 1 — Hazırlık ve repository erişimi { display-mode: "form" }
from google.colab import userdata
import os, subprocess, pathlib, shutil, time, json, zipfile, textwrap, platform, datetime, html
from IPython.display import display, HTML

REPO = "https://github.com/Jue-dev/Tessera.git"
BRANCH = "issue-6/unwind-lifecycle"
WORKDIR = pathlib.Path("/content/tessera")
LOG_DIR = pathlib.Path("/content/tessera-test-report")
LOG_DIR.mkdir(exist_ok=True)

try:
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
if not token:
    raise RuntimeError("GitHub secret GH_TOKEN bulunamadı. Colab Secrets bölümüne ekleyip notebook erişimini açın.")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
askpass = pathlib.Path("/tmp/tessera_askpass.sh")
askpass.write_text('#!/bin/sh\\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GIT_PASSWORD" ;; esac\\n')
askpass.chmod(0o700)
env = os.environ.copy()
env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "GIT_PASSWORD": token})
try:
    clone = subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO, str(WORKDIR)], env=env, capture_output=True, text=True, timeout=180)
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GIT_PASSWORD", None)
    token = None
if clone.returncode:
    safe_error = clone.stderr.replace("https://", "https://")
    raise RuntimeError("Repository indirilemedi. GH_TOKEN izinlerini ve branch adını kontrol edin.\\n" + safe_error[-3000:])
HEAD_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()
print("Hazır ✓ | Branch:", BRANCH, "| Commit:", HEAD_SHA[:12])

## Testler

Aşağıdaki otomatik testler çalıştırılır. Bir test başarısız olsa bile diğer testler devam eder; böylece tek raporda tüm sorunları görebiliriz.

In [ ]:
#@title 2 — Testleri çalıştır { display-mode: "form" }
TESTS = [
    ("GCC · strict mock", ["bash", "verify/windows_mock/run_windows_mock_tests.sh"], {}),
    ("Clang · strict mock", ["bash", "verify/windows_mock/run_windows_mock_tests.sh"], {"CXX":"clang++"}),
    ("GCC · ASan + UBSan", ["bash", "verify/windows_mock/run_windows_mock_tests.sh"], {
        "EXTRA_CXXFLAGS":"-fsanitize=address,undefined -fno-omit-frame-pointer",
        "ASAN_OPTIONS":"detect_leaks=1:halt_on_error=1",
        "UBSAN_OPTIONS":"halt_on_error=1:print_stacktrace=1"
    }),
]
records = []
for name, command, overrides in TESTS:
    started = time.time()
    test_env = os.environ.copy()
    test_env.update(overrides)
    log_file = LOG_DIR / (name.lower().replace(" · ", "_").replace(" ", "_") + ".log")
    try:
        proc = subprocess.run(command, cwd=WORKDIR, env=test_env, capture_output=True, text=True, timeout=600)
        status = "PASS" if proc.returncode == 0 and "WINDOWS_MOCK_VALIDATION_COMPLETE" in proc.stdout else "FAIL"
        combined = f"COMMAND: {' '.join(command)}\\nEXIT: {proc.returncode}\\n\\n--- STDOUT ---\\n{proc.stdout}\\n--- STDERR ---\\n{proc.stderr}"
        record = {"test":name,"status":status,"exit_code":proc.returncode,"duration_seconds":round(time.time()-started,2),"log":str(log_file)}
    except subprocess.TimeoutExpired as exc:
        status="FAIL"
        combined=f"TIMEOUT after 600 seconds\\nSTDOUT:\\n{exc.stdout}\\nSTDERR:\\n{exc.stderr}"
        record={"test":name,"status":status,"exit_code":124,"duration_seconds":600,"log":str(log_file)}
    except Exception as exc:
        status="FAIL"
        combined=f"EXCEPTION: {type(exc).__name__}: {exc}"
        record={"test":name,"status":status,"exit_code":-1,"duration_seconds":round(time.time()-started,2),"log":str(log_file)}
    log_file.write_text(combined, encoding="utf-8")
    records.append(record)
    print(("✅ " if status=="PASS" else "❌ ")+name+f" — {record['duration_seconds']} sn")
print("Test aşaması tamamlandı. Son hücre raporu oluşturur.")

## Sonuç ve paylaşılabilir hata raporu

Bu hücre; commit bilgisi, ortam, test sonuçları ve hata loglarını bir araya getirir. Rapor dosyası oluşturulur ve indirme bağlantısı sunulur.

In [ ]:
#@title 3 — Raporu hazırla ve indir { display-mode: "form" }
summary = {
    "project":"Tessera",
    "branch":BRANCH,
    "commit":HEAD_SHA,
    "commit_url":f"https://github.com/Jue-dev/Tessera/commit/{HEAD_SHA}",
    "timestamp_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "platform":platform.platform(),
    "python":platform.python_version(),
    "tests":records,
    "overall":"PASS" if records and all(x["status"]=="PASS" for x in records) else "FAIL"
}
(LOG_DIR/"summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
rows="".join(f"<tr><td>{html.escape(x['test'])}</td><td><b>{'✅ PASS' if x['status']=='PASS' else '❌ FAIL'}</b></td><td>{x['duration_seconds']} s</td><td>{x['exit_code']}</td></tr>" for x in records)
report_html=f"""
<div style='font-family:Arial,sans-serif;border:1px solid #ddd;border-radius:12px;padding:18px;max-width:900px'>
<h2 style='margin-top:0'>Tessera Portable Validation: {summary['overall']}</h2>
<p><b>Branch:</b> {html.escape(BRANCH)}<br><b>Commit:</b> <code>{HEAD_SHA}</code><br><b>Environment:</b> {html.escape(platform.platform())}</p>
<table style='border-collapse:collapse;width:100%' border='1' cellpadding='8'><tr><th>Test</th><th>Status</th><th>Duration</th><th>Exit</th></tr>{rows}</table>
<p><b>Note:</b> Portable mock tests only; Windows-native validation is not covered.</p></div>"""
display(HTML(report_html))
# Plain-text handoff report that can be pasted into a chat
handoff=["TESSERA TEST REPORT",f"Overall: {summary['overall']}",f"Branch: {BRANCH}",f"Commit: {HEAD_SHA}",f"Commit URL: {summary['commit_url']}",f"Timestamp UTC: {summary['timestamp_utc']}",f"Environment: {summary['platform']}",""]
for item in records:
    handoff += [f"[{item['status']}] {item['test']} | exit={item['exit_code']} | {item['duration_seconds']}s", "Log: "+item['log']]
    if item['status'] != 'PASS':
        handoff += ["--- FAILURE LOG ---",(pathlib.Path(item['log']).read_text(encoding='utf-8',errors='replace')[-12000:]),"--- END LOG ---"]
handoff_text="\\n".join(handoff)
(LOG_DIR/"CHAT_REPORT.txt").write_text(handoff_text,encoding="utf-8")
archive=pathlib.Path("/content/Tessera_Test_Report.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for file in LOG_DIR.glob("*"):
        z.write(file,arcname=file.name)
print("\\n--- CHAT'E KOPYALANACAK RAPOR ---\\n")
print(handoff_text)
print("\\nZIP raporu:",archive)
try:
    from google.colab import files
    files.download(str(archive))
except Exception as e:
    print("Otomatik indirme başlatılamadı. Dosya yolu:",archive,"|",e)
if summary['overall'] != 'PASS':
    print("\\nTestlerde hata var. Yukarıdaki rapor ve ZIP dosyasını bu sohbete yükleyin.")

---

### Bana nasıl aktaracaksın?
- Testler tamamlanınca ekranda çıkan **CHAT'E KOPYALANACAK RAPOR** kısmını kopyalayıp buraya yapıştır; veya
- İndirilen `Tessera_Test_Report.zip` dosyasını doğrudan sohbete yükle.

Rapor commit SHA içerdiğinden hangi kodun test edildiğini kesin olarak ayırt edebilirim. Token rapora dahil edilmez.

In [ ]:
#@title (Gizli) Notebook yardımcı kodları
# Bu hücre kullanıcı arayüzü için teknik yardımcılar içerir; çalıştırmak gerekmez.
